# Performance monitoring

F-tests, exception counts and shortfalls for each quarter after the sample period.

**Weights:** these reports now use each factor's configured weights throughout. The original
notebook used equal weights for the F-test and the exception backtests (but configured weights
for the VaR itself). Pass `use_config_weights=False` to reproduce the original numbers.

Settings come from `configs/var_config_volfactor.yml`, found automatically in this folder or
any parent folder. To use a different config or price file (same format, different data):

```python
vf.set_config("path/to/config.yml")
vf.set_data("path/to/prices.csv")   # or a .parquet file, or a DataFrame
```

In [ ]:
import volatility_factor as vf
from volatility_factor import monitoring

cfg = vf.get_config()
# Full sample, its last year, and quarters Q1-Q5 after the sample end.
periods = vf.MonitoringPeriods.from_config(cfg)
periods

## 1. F-Test

In [ ]:
output_name = "FStats_20250930.xlsx"
save_output = True

fstats = monitoring.f_test_report(periods, quarters=[1, 2, 3])
if save_output:
    fstats.to_excel(cfg.output_dir / output_name, index=True)
fstats

## 2. Exceptions
### 2.1 One-day exceptions, by quarter

In [ ]:
output_name = "OneDayExceptions_20250630.xlsx"
save_output = True

one_day = monitoring.one_day_exceptions(periods, quarters=[1, 2, 3, 4], nsims=10_000)
if save_output:
    one_day.to_excel(cfg.output_dir / output_name, index=False)
one_day

### 2.2 N-day exceptions

Exceptions from the start of Q2 through the end of `quarter` (3, 4 or 5), with the
independence test and clustering adjustment.

In [ ]:
quarter = 3
output_name = "Exceptions_20250930.xlsx"
save_output = True

nday = monitoring.nday_exceptions(periods, quarter=quarter, nsims=10_000)
if save_output:
    nday.to_excel(cfg.output_dir / output_name, index=False)
nday

### 2.3 Shortfalls for one index

In [ ]:
quarter = 3
ticker = "G0BA"
weights = None
ndays = 10
output_name = f"{ticker}_{weights}_{ndays}_shortfall_20230930.xlsx"
save_output = True

shortfall = monitoring.shortfall_report(ticker, periods, quarter=quarter, weights=weights,
                                        ndays=ndays, start_vals=("2022-01-01", 0.50))
if save_output:
    shortfall.to_excel(cfg.output_dir / output_name, index=True)
shortfall